# 覓聲 MeetSum — 把摘要微調版放進 Ollama
1. 下載原始權重 → 2. 重新訓練摘要 LoRA（和上次相同的資料與設定，約 31 分）→ 3. 把 LoRA 加回原始權重 → 4. 匯入 Ollama（模型名稱 `meetsum-summary`）
→ 5. 用覓聲正式的呼叫方式（/api/chat、format=json）比較原版與微調版 → 6. 下載結果。不需要雲端硬碟。原版 qwen3.8:27b 不動。

In [ ]:
# 0. 環境：程式與測試資料都內嵌在 notebook 裡；安裝並啟動 Ollama，背景下載 qwen3.8:27b
import os, sys, json, gzip, base64, subprocess, shutil, time, glob, hashlib
W = '/content/deploy'; os.makedirs(W, exist_ok=True)
for n, s in {'merge_lora.py': '"""把 LoRA 直接加回原始權重檔（逐個 safetensors 分片處理）。\n\n不經過 transformers 的 save_pretrained：輸出的檔名、分片、張量名稱、dtype 和官方 Qwen/Qwen3.8-27B 完全一樣，\n只有被 LoRA 改到的 400 個權重數值不同。這樣 Ollama / llama.cpp 讀到的就是「官方格式」，避免轉換器不認得。\n\nW_new = W + (B @ A) * (alpha / r)\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport shutil\nfrom pathlib import Path\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--adapter", required=True)\n    ap.add_argument("--out", required=True)\n    a = ap.parse_args()\n\n    import torch\n    from safetensors import safe_open\n    from safetensors.torch import load_file, save_file\n\n    base, out = Path(a.base), Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    cfg = json.loads((Path(a.adapter) / "adapter_config.json").read_text())\n    scale = cfg["lora_alpha"] / cfg["r"]\n    lora = load_file(str(Path(a.adapter) / "adapter_model.safetensors"))\n    mods = sorted({k[: k.index(".lora_A")] for k in lora if ".lora_A" in k})\n    print(f"LoRA：{len(mods)} 個模組，r={cfg[\'r\']}，alpha={cfg[\'lora_alpha\']}，scale={scale}", flush=True)\n\n    index = json.loads((base / "model.safetensors.index.json").read_text())\n    wmap = index["weight_map"]\n    keys = list(wmap)\n\n    def find(mod: str) -> str:\n        m = mod.removeprefix("base_model.model.")\n        if m + ".weight" in wmap:\n            return m + ".weight"\n        tail = re.search(r"layers\\.\\d+\\..*$", m).group(0) + ".weight"\n        hits = [k for k in keys if k.endswith(tail) and "visual" not in k and "vision" not in k and "mtp" not in k]\n        if len(hits) != 1:\n            raise RuntimeError(f"{mod} 對不到唯一的原始權重：{hits[:5]}")\n        return hits[0]\n\n    target = {find(m): m for m in mods}\n    print("對應範例：", list(target.items())[:2], flush=True)\n    by_shard: dict[str, list[str]] = {}\n    for k in target:\n        by_shard.setdefault(wmap[k], []).append(k)\n\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    done = 0\n    for shard in sorted(set(wmap.values())):\n        src = base / shard\n        if shard not in by_shard:\n            shutil.copy(src, out / shard)\n            continue\n        with safe_open(str(src), "pt") as f:\n            meta = f.metadata()\n        t = load_file(str(src))\n        for k in by_shard[shard]:\n            m = target[k]\n            A = lora[m + ".lora_A.weight"].to(dev, torch.float32)\n            B = lora[m + ".lora_B.weight"].to(dev, torch.float32)\n            w = t[k]\n            delta = (B @ A) * scale\n            if delta.shape != w.shape:\n                raise RuntimeError(f"{k} 形狀不合：{tuple(w.shape)} vs {tuple(delta.shape)}")\n            t[k] = (w.to(dev, torch.float32) + delta).to(w.dtype).cpu()\n            done += 1\n        save_file(t, str(out / shard), metadata=meta)\n        print(f"  {shard}：合併 {len(by_shard[shard])} 個權重", flush=True)\n        del t\n    for f in base.iterdir():\n        if f.is_file() and not f.name.endswith(".safetensors") and not (out / f.name).exists():\n            shutil.copy(f, out / f.name)\n    if done != len(mods):\n        raise RuntimeError(f"只合併了 {done}/{len(mods)} 個")\n    print(f"完成：合併 {done} 個權重 → {out}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'ollama_eval.py': '"""用和覓聲正式系統相同的方式呼叫 Ollama（/api/chat、format=json、temperature 0、seed 42），\n在 8 場測試會議上比較：\n- 原版 qwen3.8:27b（思考模式開啟，= 目前 .env 的 LLM_THINK=True）\n- 微調版（思考模式關閉，= 訓練時的設定）\n分數用和 Colab 評測相同的 eval_sum.score。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport requests\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport eval_sum as E  # noqa: E402\n\n\ndef call(host: str, model: str, ex: dict, think: bool, num_ctx: int) -> tuple[str, dict]:\n    payload = {\n        "model": model,\n        "messages": [{"role": "system", "content": ex["system"]}, {"role": "user", "content": ex["user"]}],\n        "stream": False,\n        "format": "json",\n        "options": {"temperature": 0, "seed": 42, "num_ctx": num_ctx},\n        "think": think,\n    }\n    t0 = time.time()\n    r = requests.post(f"{host}/api/chat", json=payload, timeout=3600)\n    if r.status_code != 200:\n        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")\n    body = r.json()\n    msg = body.get("message") or {}\n    stat = {"secs": round(time.time() - t0, 1), "prompt_tokens": body.get("prompt_eval_count"),\n            "out_tokens": body.get("eval_count"), "thinking_chars": len(msg.get("thinking") or ""),\n            "done_reason": body.get("done_reason")}\n    return msg.get("content") or "", stat\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True, help="sft_test.jsonl")\n    ap.add_argument("--host", default="http://127.0.0.1:11434")\n    ap.add_argument("--model", required=True)\n    ap.add_argument("--tag", required=True)\n    ap.add_argument("--think", choices=["true", "false"], required=True)\n    ap.add_argument("--num-ctx", type=int, default=40960)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--limit", type=int, default=0)\n    a = ap.parse_args()\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    rows = [json.loads(x) for x in open(a.data, encoding="utf-8") if x.strip()]\n    if a.limit:\n        rows = rows[: a.limit]\n    pred = out / f"{a.tag}.predictions.jsonl"\n    done = {}\n    if pred.exists():\n        done = {json.loads(l)["id"]: json.loads(l) for l in pred.open(encoding="utf-8")}\n    with pred.open("a", encoding="utf-8") as f:\n        for r in rows:\n            if r["id"] in done:\n                continue\n            reply, stat = call(a.host, a.model, r, a.think == "true", a.num_ctx)\n            rec = {"id": r["id"], "reply": reply, **stat}\n            done[r["id"]] = rec\n            f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            f.flush()\n            print(f"  {a.tag} {r[\'id\']}  {stat}", flush=True)\n    per = []\n    for r in rows:\n        s = E.score(done[r["id"]]["reply"], json.loads(r["target"]), r["transcript"])\n        s.update(id=r["id"], lang=r["lang"], secs=done[r["id"]]["secs"], out_tokens=done[r["id"]]["out_tokens"])\n        per.append(s)\n    rep = {"all": E.summarize(per), "zh": E.summarize([p for p in per if p["lang"] == "zh"]),\n           "en": E.summarize([p for p in per if p["lang"] == "en"]), "per_meeting": per,\n           "avg_secs": round(sum(p["secs"] for p in per) / len(per), 1)}\n    (out / f"{a.tag}.report.json").write_text(json.dumps(rep, ensure_ascii=False, indent=1), encoding="utf-8")\n    print(a.tag, json.dumps(rep["all"], ensure_ascii=False), "平均秒數", rep["avg_secs"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'eval_sum.py': '"""摘要評測：原版（不掛 LoRA）vs 微調版（掛 LoRA），和我寫的參考摘要比較。\n\n指標（每場會議算完取平均）：\n- json_ok：輸出是合法 JSON 且結構正確\n- quote_ok：引用原文確實出現在逐字稿的比例（有沒有捏造）\n- count_diff：四類項目（報告／討論／決議／待辦）數量和參考摘要的平均差距（越小越好）\n- topic_f1：報告＋討論的議題，和參考摘要逐一配對的 F1（字元相似度 ≥ 0.5 算配對成功）\n- action_f1：待辦事項和參考摘要配對的 F1\n- text_sim：整份摘要文字（headline＋各項摘要＋待辦）和參考摘要的字元 ROUGE-L F1\n- headline_sim：一句話總結的字元 ROUGE-L F1\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport difflib\nimport json\nimport re\nimport sys\nimport time\nfrom pathlib import Path\n\nCATS = ["reports", "discussions", "decisions", "action_items"]\n\n\ndef parse(text: str):\n    t = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()\n    t = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", t)\n    try:\n        j = json.loads(t)\n    except Exception:\n        m = re.search(r"\\{.*\\}", t, flags=re.S)\n        if not m:\n            return None\n        try:\n            j = json.loads(m.group(0))\n        except Exception:\n            return None\n    if not isinstance(j, dict) or not isinstance(j.get("meeting_content"), dict):\n        return None\n    return j\n\n\ndef _items(j, cat):\n    v = (j or {}).get("meeting_content", {}).get(cat) or []\n    return [x for x in v if isinstance(x, dict)]\n\n\ndef lcs_f1(a: str, b: str) -> float:\n    if not a or not b:\n        return 0.0\n    a, b = a[:1500], b[:1500]\n    prev = [0] * (len(b) + 1)\n    for x in a:\n        cur = [0]\n        for j, y in enumerate(b, 1):\n            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))\n        prev = cur\n    l = prev[-1]\n    p, r = l / len(a), l / len(b)\n    return 2 * p * r / (p + r) if p + r else 0.0\n\n\ndef match_f1(pred: list[str], ref: list[str], thr=0.5) -> float:\n    if not pred and not ref:\n        return 1.0\n    if not pred or not ref:\n        return 0.0\n    used, tp = set(), 0\n    for p in pred:\n        best, bi = 0, None\n        for i, r in enumerate(ref):\n            if i in used:\n                continue\n            s = difflib.SequenceMatcher(None, p.lower(), r.lower()).ratio()\n            if s > best:\n                best, bi = s, i\n        if best >= thr:\n            used.add(bi)\n            tp += 1\n    pr, rc = tp / len(pred), tp / len(ref)\n    return 2 * pr * rc / (pr + rc) if pr + rc else 0.0\n\n\ndef flat_text(j) -> str:\n    parts = [str((j or {}).get("headline") or "")]\n    for cat in CATS:\n        for it in _items(j, cat):\n            parts.append(str(it.get("topic") or ""))\n            parts.append(str(it.get("summary") or it.get("task") or ""))\n            parts += [str(k) for k in (it.get("key_points") or [])]\n    return "".join(parts)\n\n\ndef score(pred_text: str, ref: dict, transcript: dict) -> dict:\n    j = parse(pred_text)\n    if j is None:\n        return {"json_ok": 0, "quote_ok": 0, "count_diff": None, "topic_f1": 0, "action_f1": 0, "text_sim": 0, "headline_sim": 0}\n    seg = {s["segment_id"]: s["text"] for s in transcript["segments"]}\n    q_tot = q_ok = 0\n    for cat in CATS:\n        for it in _items(j, cat):\n            for e in it.get("evidence") or []:\n                if not isinstance(e, dict):\n                    continue\n                q_tot += 1\n                q_ok += bool(e.get("quote")) and str(e.get("quote")) in seg.get(e.get("segment_id"), "")\n    topics = lambda x: [str(i.get("topic") or "") for c in ("reports", "discussions") for i in _items(x, c)]\n    tasks = lambda x: [str(i.get("task") or "") for i in _items(x, "action_items")]\n    return {\n        "json_ok": 1,\n        "quote_ok": q_ok / q_tot if q_tot else 0.0,\n        "count_diff": sum(abs(len(_items(j, c)) - len(_items(ref, c))) for c in CATS) / 4,\n        "topic_f1": match_f1(topics(j), topics(ref)),\n        "action_f1": match_f1(tasks(j), tasks(ref)),\n        "text_sim": lcs_f1(flat_text(j), flat_text(ref)),\n        "headline_sim": lcs_f1(str(j.get("headline") or ""), str(ref.get("headline") or "")),\n    }\n\n\ndef summarize(rows: list[dict]) -> dict:\n    out = {"n": len(rows)}\n    for k in ["json_ok", "quote_ok", "count_diff", "topic_f1", "action_f1", "text_sim", "headline_sim"]:\n        v = [r[k] for r in rows if r[k] is not None]\n        out[k] = round(sum(v) / len(v), 4) if v else None\n    return out\n\n\nclass Shared:\n    def __init__(self, base, adapter, load_4bit=False):\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        import lm_utils as L\n        self.L = L\n        self.tok = L.load_tokenizer(base)\n        self.tok.padding_side = "left"\n        self.model = L.load_model(base, adapter, load_4bit=load_4bit)\n        self.use_adapter = True\n\n    def generate(self, batch, max_new_tokens=5120):\n        import torch\n        texts = [self.L.chat_prompt(self.tok, ex["system"], ex["user"]) for ex in batch]\n        enc = self.tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)\n        ctx = contextlib.nullcontext() if self.use_adapter else self.model.disable_adapter()\n        with torch.no_grad(), ctx:\n            gen = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=self.tok.pad_token_id)\n        return self.tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)\n\n\ndef run(data: Path, base: str, adapter: str, out: Path, batch_size: int, load_4bit: bool):\n    out.mkdir(parents=True, exist_ok=True)\n    rows = [json.loads(x) for x in (data / "sft_test.jsonl").open(encoding="utf-8")]\n    m = Shared(base, adapter, load_4bit)\n    report = {}\n    for tag, use in [("base", False), ("ft", True)]:\n        m.use_adapter = use\n        pred_path = out / f"sum_{tag}.predictions.jsonl"\n        done = {}\n        if pred_path.exists():\n            done = {json.loads(l)["id"]: json.loads(l)["reply"] for l in pred_path.open(encoding="utf-8")}\n        todo = [r for r in rows if r["id"] not in done]\n        t0 = time.time()\n        with pred_path.open("a", encoding="utf-8") as f:\n            for i in range(0, len(todo), batch_size):\n                b = todo[i:i + batch_size]\n                for r, rep in zip(b, m.generate(b)):\n                    done[r["id"]] = rep\n                    f.write(json.dumps({"id": r["id"], "reply": rep}, ensure_ascii=False) + "\\n")\n                print(f"  {tag} {i + len(b)}/{len(todo)}  {time.time() - t0:.0f}s", flush=True)\n        per = []\n        for r in rows:\n            s = score(done[r["id"]], json.loads(r["target"]), r["transcript"])\n            s["id"], s["lang"] = r["id"], r["lang"]\n            per.append(s)\n        report[tag] = {"all": summarize(per), "zh": summarize([p for p in per if p["lang"] == "zh"]),\n                       "en": summarize([p for p in per if p["lang"] == "en"]), "per_meeting": per}\n        (out / "sum_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")\n        print(tag, json.dumps(report[tag]["all"], ensure_ascii=False), flush=True)\n    return report\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--adapter", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--batch-size", type=int, default=4)\n    ap.add_argument("--load-4bit", action="store_true")\n    a = ap.parse_args()\n    run(Path(a.data), a.base, a.adapter, Path(a.out), a.batch_size, a.load_4bit)\n', 'lm_utils.py': '"""訓練與評測共用：組出和推論時完全相同的 prompt 文字（關閉 thinking）。"""\nfrom __future__ import annotations\n\n\ndef chat_prompt(tok, system: str, user: str) -> str:\n    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]\n    try:\n        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    except TypeError:\n        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n\ndef end_text(tok) -> str:\n    """回答結尾：Qwen 系列是 <|im_end|>。"""\n    eos = tok.eos_token or "<|im_end|>"\n    return "<|im_end|>" if "<|im_end|>" in tok.get_vocab() else eos\n\n\ndef load_tokenizer(path: str):\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\n\ndef load_model(path: str, adapter: str | None = None, dtype: str = "bfloat16", load_4bit: bool = False):\n    import torch\n    from transformers import AutoConfig\n    cfg = AutoConfig.from_pretrained(path, trust_remote_code=True)\n    kw = {"dtype": getattr(torch, dtype), "device_map": "auto", "trust_remote_code": True, "low_cpu_mem_usage": True}\n    if load_4bit:\n        from transformers import BitsAndBytesConfig\n        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                                       bnb_4bit_compute_dtype=getattr(torch, dtype),\n                                                       bnb_4bit_use_double_quant=True)\n    if getattr(cfg, "vision_config", None) is not None:\n        from transformers import AutoModelForImageTextToText as Auto\n    else:\n        from transformers import AutoModelForCausalLM as Auto\n    model = Auto.from_pretrained(path, **kw)\n    if adapter:\n        from peft import PeftModel\n        model = PeftModel.from_pretrained(model, adapter)\n    model.eval()\n    return model\n', 'train_lora.py': '"""步驟 3（Colab GPU）：LoRA 微調 Qwen3.8-27B 的錯字校正能力。\n\n- 只訓練語言模型的注意力與 MLP 投影層（LoRA），原始權重不動；視覺部分完全不碰。\n- loss 只算「模型的回答」那段，prompt（系統指令＋逐字稿）不算。\n- checkpoint 存到 --out（請放在 Google Drive），斷線後重跑會自動接續。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport random\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport lm_utils as L  # noqa: E402\n\n# 全注意力層：q/k/v/o；線性注意力層（Gated DeltaNet）：in_proj_qkv / in_proj_z / out_proj；每層的 MLP。\n_MODS = (r"(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|linear_attn\\.(in_proj_qkv|in_proj_z|out_proj)"\n         r"|mlp\\.(gate_proj|up_proj|down_proj))$")\nTARGET = r"^(base_model\\.model\\.)?model\\.(language_model\\.)?layers\\.\\d+\\." + _MODS\n\n\ndef encode(tok, ex: dict, max_len: int):\n    prompt = L.chat_prompt(tok, ex["system"], ex["user"])\n    answer = ex["target"] + L.end_text(tok) + "\\n"\n    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]\n    a_ids = tok(answer, add_special_tokens=False)["input_ids"]\n    if len(p_ids) + len(a_ids) > max_len:\n        return None\n    return {"input_ids": p_ids + a_ids, "labels": [-100] * len(p_ids) + a_ids}\n\n\nclass Collator:\n    def __init__(self, pad_id: int):\n        self.pad_id = pad_id\n\n    def __call__(self, feats):\n        import torch\n        n = max(len(f["input_ids"]) for f in feats)\n        ids = torch.full((len(feats), n), self.pad_id, dtype=torch.long)\n        lab = torch.full((len(feats), n), -100, dtype=torch.long)\n        att = torch.zeros((len(feats), n), dtype=torch.long)\n        for i, f in enumerate(feats):\n            k = len(f["input_ids"])\n            ids[i, :k] = torch.tensor(f["input_ids"])\n            lab[i, :k] = torch.tensor(f["labels"])\n            att[i, :k] = 1\n        return {"input_ids": ids, "labels": lab, "attention_mask": att}\n\n\ndef make_trainer_cls(Trainer, TrainerCallback):\n    """長輸入省記憶體：不產生整段的 logits，只對「回答」那段分塊算 lm_head + loss（每塊用 checkpoint 重算）。"""\n    import torch\n    import torch.nn.functional as F\n    from torch.utils.checkpoint import checkpoint\n\n    def _ce(h, lab, head):\n        return F.cross_entropy(head(h).float().reshape(-1, head.out_features), lab.reshape(-1),\n                               ignore_index=-100, reduction="sum")\n\n    class AnswerOnlyTrainer(Trainer):\n        def __init__(self, *a, **k):\n            super().__init__(*a, **k)\n            self.model_accepts_loss_kwargs = False  # 每筆取平均，Trainer 再除以 grad_accum\n\n        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n            labels = inputs["labels"]\n            n = labels.shape[1]\n            valid = labels != -100\n            first = int(valid.float().argmax(dim=1).min().item()) if valid.any() else n - 1\n            first = max(1, first)\n            base = model.get_base_model() if hasattr(model, "get_base_model") else model\n            hid = base.model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],\n                             use_cache=False).last_hidden_state\n            h = hid[:, first - 1:n - 1]\n            lab = labels[:, first:n]\n            head = base.lm_head\n            tot = torch.zeros((), device=h.device, dtype=torch.float32)\n            for i in range(0, h.shape[1], 1024):\n                tot = tot + checkpoint(_ce, h[:, i:i + 1024], lab[:, i:i + 1024], head, use_reentrant=False)\n            loss = tot / (lab != -100).sum().clamp(min=1)\n            return (loss, None) if return_outputs else loss\n\n        def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):\n            with torch.no_grad():\n                loss = self.compute_loss(model, inputs)\n            return (loss.detach(), None, None)\n\n    class MemCallback(TrainerCallback):\n        def on_step_end(self, args, state, control, **kw):\n            if state.global_step <= 2 or state.global_step % 5 == 0:\n                print(f"[step {state.global_step}] 顯示卡記憶體峰值 {torch.cuda.max_memory_allocated() / 2**30:.1f} GB", flush=True)\n\n    return AnswerOnlyTrainer, MemCallback\n\ndef load(path: Path, kinds: set[str] | None = None) -> list[dict]:\n    rows = [json.loads(x) for x in path.open(encoding="utf-8") if x.strip()]\n    return [r for r in rows if kinds is None or r["kind"] in kinds]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--epochs", type=float, default=2.0)\n    ap.add_argument("--lr", type=float, default=1e-4)\n    ap.add_argument("--rank", type=int, default=16)\n    ap.add_argument("--alpha", type=int, default=32)\n    ap.add_argument("--max-len", type=int, default=6144)\n    ap.add_argument("--grad-accum", type=int, default=16)\n    ap.add_argument("--save-steps", type=int, default=20)\n    ap.add_argument("--dev-limit", type=int, default=48)\n    ap.add_argument("--load-4bit", action="store_true", help="顯示卡記憶體不夠時改用 QLoRA")\n    ap.add_argument("--max-steps", type=int, default=-1, help="測試用")\n    ap.add_argument("--max-train", type=int, default=1200, help="訓練樣本上限（控制 Colab 時數）；0 = 全部")\n    a = ap.parse_args()\n\n    # Colab 內建的舊版 torchao（0.10）會讓新版 peft 直接報錯；本專案用不到它，有舊版就移除\n    try:\n        import importlib.metadata as md\n        from packaging.version import Version\n        if Version(md.version("torchao")) < Version("0.16.0"):\n            import subprocess\n            print("移除不相容的舊版 torchao", md.version("torchao"), flush=True)\n            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)\n    except Exception:\n        pass\n\n    import os\n    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n    import torch\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n    from transformers import Trainer, TrainerCallback, TrainingArguments, trainer_utils\n\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    tok = L.load_tokenizer(a.base)\n\n    train_rows = load(Path(a.data) / "sft_train.jsonl")\n    dev_rows = load(Path(a.data) / "sft_dev.jsonl")\n    random.Random(0).shuffle(dev_rows)\n    random.Random(1).shuffle(train_rows)\n    if a.max_train:\n        train_rows = train_rows[:a.max_train]\n    train = [e for e in (encode(tok, r, a.max_len) for r in train_rows) if e]\n    dev = [e for e in (encode(tok, r, a.max_len) for r in dev_rows) if e][:a.dev_limit]\n    lens = sorted(len(e["input_ids"]) for e in train)\n    print(f"訓練 {len(train)} 筆（超過 {a.max_len} token 丟掉 {len(train_rows) - len(train)} 筆）；"\n          f"dev {len(dev)} 筆；長度中位數 {lens[len(lens) // 2]}、最大 {lens[-1]}", flush=True)\n\n    model = L.load_model(a.base, load_4bit=a.load_4bit)\n    model.config.use_cache = False\n    if a.load_4bit:\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    else:\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.enable_input_require_grads()\n    lcfg = LoraConfig(r=a.rank, lora_alpha=a.alpha, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",\n                      target_modules=TARGET)\n    model = get_peft_model(model, lcfg)\n    model.print_trainable_parameters()\n    gb = model.get_base_model()\n    gb.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    gb.enable_input_require_grads()\n    n_ckpt = sum(1 for m in gb.modules() if getattr(m, "gradient_checkpointing", False))\n    print("gradient checkpointing 開啟的模組數：", n_ckpt, flush=True)\n    names = [n for n, _ in model.named_modules() if n.endswith("lora_A")]\n    print("LoRA 掛在", len(names), "個模組，例如", names[:3], flush=True)\n    if not names:\n        raise RuntimeError("LoRA 沒有掛上任何模組，請檢查 TARGET 正規表示式")\n\n    steps_per_epoch = math.ceil(len(train) / a.grad_accum)\n    total_steps = a.max_steps if a.max_steps > 0 else math.ceil(steps_per_epoch * a.epochs)\n    args = TrainingArguments(\n        output_dir=str(out / "checkpoints"),\n        per_device_train_batch_size=1, per_device_eval_batch_size=1,\n        gradient_accumulation_steps=a.grad_accum,\n        num_train_epochs=a.epochs, max_steps=a.max_steps,\n        learning_rate=a.lr, lr_scheduler_type="cosine", warmup_steps=max(1, int(total_steps * 0.03)), weight_decay=0.0,\n        bf16=torch.cuda.is_available(), logging_steps=2,\n        eval_strategy="steps" if dev else "no", eval_steps=a.save_steps, save_strategy="steps", save_steps=a.save_steps,\n        save_total_limit=3, load_best_model_at_end=bool(dev), metric_for_best_model="eval_loss", greater_is_better=False,\n        report_to="none", remove_unused_columns=False, dataloader_num_workers=0, label_names=["labels"],\n        gradient_checkpointing=False,  # 上面已經自己開好\n        seed=42,\n    )\n    print(f"每個 epoch {steps_per_epoch} 步，共 {a.epochs} 個 epoch", flush=True)\n    TCls, MemCb = make_trainer_cls(Trainer, TrainerCallback)\n    trainer = TCls(model=model, args=args, train_dataset=train, eval_dataset=dev or None,\n                   data_collator=Collator(tok.pad_token_id), callbacks=[MemCb()])\n    last = trainer_utils.get_last_checkpoint(str(out / "checkpoints")) if (out / "checkpoints").exists() else None\n    if last:\n        print("接續上次的 checkpoint：", last, flush=True)\n    trainer.train(resume_from_checkpoint=last)\n    final = out / "final"\n    trainer.model.save_pretrained(str(final))\n    tok.save_pretrained(str(final))\n    hist = [h for h in trainer.state.log_history]\n    (out / "train_log.json").write_text(json.dumps(hist, ensure_ascii=False, indent=1), encoding="utf-8")\n    (out / "train_args.json").write_text(json.dumps(vars(a), ensure_ascii=False, indent=1), encoding="utf-8")\n    print("完成，adapter 存在", final, "（最佳 checkpoint：", trainer.state.best_model_checkpoint, "）")\n\n\nif __name__ == "__main__":\n    main()\n'}.items():
    open(f'{W}/{n}', 'w', encoding='utf-8').write(s)
os.makedirs(f'{W}/data', exist_ok=True)
for n, b in {'sft_train.jsonl': '...', 'sft_dev.jsonl': '...'}.items():
    open(f'{W}/data/{n}', 'wb').write(gzip.decompress(base64.b64decode(b)))
open(f'{W}/test8.jsonl', 'wb').write(gzip.decompress(base64.b64decode('...')))
open(f'{W}/hf_sum_report.json', 'w', encoding='utf-8').write('{\n "base": {\n  "all": {\n   "n": 8,\n   "json_ok": 0.625,\n   "quote_ok": 0.605,\n   "count_diff": 1.8,\n   "topic_f1": 0.3086,\n   "action_f1": 0.1944,\n   "text_sim": 0.2494,\n   "headline_sim": 0.2161\n  },\n  "zh": {\n   "n": 5,\n   "json_ok": 0.4,\n   "quote_ok": 0.393,\n   "count_diff": 2.375,\n   "topic_f1": 0.1748,\n   "action_f1": 0.1111,\n   "text_sim": 0.1091,\n   "headline_sim": 0.0653\n  },\n  "en": {\n   "n": 3,\n   "json_ok": 1.0,\n   "quote_ok": 0.9583,\n   "count_diff": 1.4167,\n   "topic_f1": 0.5317,\n   "action_f1": 0.3333,\n   "text_sim": 0.4833,\n   "headline_sim": 0.4675\n  },\n  "per_meeting": [\n   {\n    "json_ok": 0,\n    "quote_ok": 0,\n    "count_diff": null,\n    "topic_f1": 0,\n    "action_f1": 0,\n    "text_sim": 0,\n    "headline_sim": 0,\n    "id": "L_R003S01C02",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9649122807017544,\n    "count_diff": 1.5,\n    "topic_f1": 0.5882352941176471,\n    "action_f1": 0.3333333333333333,\n    "text_sim": 0.3518808777429467,\n    "headline_sim": 0.16867469879518074,\n    "id": "L_R003S02C02",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 0,\n    "quote_ok": 0,\n    "count_diff": null,\n    "topic_f1": 0,\n    "action_f1": 0,\n    "text_sim": 0,\n    "headline_sim": 0,\n    "id": "L_R004S01C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 0,\n    "quote_ok": 0,\n    "count_diff": null,\n    "topic_f1": 0,\n    "action_f1": 0,\n    "text_sim": 0,\n    "headline_sim": 0,\n    "id": "L_R004S02C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 3.25,\n    "topic_f1": 0.28571428571428575,\n    "action_f1": 0.22222222222222224,\n    "text_sim": 0.19367588932806326,\n    "headline_sim": 0.15789473684210528,\n    "id": "L_R004S03C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 1.0,\n    "topic_f1": 0.6666666666666666,\n    "action_f1": 0.0,\n    "text_sim": 0.5286666666666666,\n    "headline_sim": 0.5224489795918368,\n    "id": "EN2002a",\n    "lang": "en"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.875,\n    "count_diff": 1.5,\n    "topic_f1": 0.42857142857142855,\n    "action_f1": 0.5,\n    "text_sim": 0.4713333333333333,\n    "headline_sim": 0.430188679245283,\n    "id": "EN2002b",\n    "lang": "en"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 1.75,\n    "topic_f1": 0.5000000000000001,\n    "action_f1": 0.5,\n    "text_sim": 0.45,\n    "headline_sim": 0.4497607655502392,\n    "id": "EN2002c",\n    "lang": "en"\n   }\n  ]\n },\n "ft": {\n  "all": {\n   "n": 8,\n   "json_ok": 1.0,\n   "quote_ok": 0.9854,\n   "count_diff": 1.625,\n   "topic_f1": 0.6262,\n   "action_f1": 0.2684,\n   "text_sim": 0.4706,\n   "headline_sim": 0.4222\n  },\n  "zh": {\n   "n": 5,\n   "json_ok": 1.0,\n   "quote_ok": 0.9853,\n   "count_diff": 1.95,\n   "topic_f1": 0.6442,\n   "action_f1": 0.3495,\n   "text_sim": 0.4162,\n   "headline_sim": 0.3837\n  },\n  "en": {\n   "n": 3,\n   "json_ok": 1.0,\n   "quote_ok": 0.9855,\n   "count_diff": 1.0833,\n   "topic_f1": 0.5961,\n   "action_f1": 0.1333,\n   "text_sim": 0.5613,\n   "headline_sim": 0.4863\n  },\n  "per_meeting": [\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 2.25,\n    "topic_f1": 0.75,\n    "action_f1": 0.0,\n    "text_sim": 0.410323709536308,\n    "headline_sim": 0.35416666666666663,\n    "id": "L_R003S01C02",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 0.75,\n    "topic_f1": 0.823529411764706,\n    "action_f1": 0.0,\n    "text_sim": 0.5748953974895398,\n    "headline_sim": 0.4444444444444445,\n    "id": "L_R003S02C02",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9859154929577465,\n    "count_diff": 2.75,\n    "topic_f1": 0.6875,\n    "action_f1": 0.6666666666666665,\n    "text_sim": 0.42266666666666663,\n    "headline_sim": 0.3466666666666666,\n    "id": "L_R004S01C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9629629629629629,\n    "count_diff": 2.25,\n    "topic_f1": 0.39999999999999997,\n    "action_f1": 0.6363636363636364,\n    "text_sim": 0.29577956051621906,\n    "headline_sim": 0.34146341463414637,\n    "id": "L_R004S02C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9777777777777777,\n    "count_diff": 1.75,\n    "topic_f1": 0.5599999999999999,\n    "action_f1": 0.4444444444444445,\n    "text_sim": 0.3771661569826707,\n    "headline_sim": 0.4318181818181818,\n    "id": "L_R004S03C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9565217391304348,\n    "count_diff": 1.25,\n    "topic_f1": 0.6666666666666667,\n    "action_f1": 0.0,\n    "text_sim": 0.6126666666666667,\n    "headline_sim": 0.43181818181818177,\n    "id": "EN2002a",\n    "lang": "en"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 0.5,\n    "topic_f1": 0.5333333333333333,\n    "action_f1": 0.4,\n    "text_sim": 0.572,\n    "headline_sim": 0.555956678700361,\n    "id": "EN2002b",\n    "lang": "en"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 1.5,\n    "topic_f1": 0.5882352941176471,\n    "action_f1": 0.0,\n    "text_sim": 0.49933333333333335,\n    "headline_sim": 0.47104247104247104,\n    "id": "EN2002c",\n    "lang": "en"\n   }\n  ]\n }\n}')
print('測試會議', sum(1 for _ in open(f'{W}/test8.jsonl', encoding='utf-8')), '場')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'失敗 exit {p.returncode}: {cmd}')
    return p.returncode
if shutil.which('ollama') is None:
    sh('apt-get install -y -qq zstd pciutils lshw > /content/apt.log 2>&1', check=False)
    sh('curl -fsSL https://ollama.com/install.sh | sh > /content/ollama_install.log 2>&1 || (tail -20 /content/ollama_install.log; false)')
sh(f'{sys.executable} -m pip uninstall -y -q torchao', check=False)
sh(f'{sys.executable} -m pip install -q -U transformers peft accelerate bitsandbytes flash-linear-attention', check=False)
os.environ['OLLAMA_KEEP_ALIVE'] = '24h'
if subprocess.run('curl -sf http://127.0.0.1:11434/api/version', shell=True).returncode != 0:
    subprocess.Popen('nohup ollama serve > /content/ollama.log 2>&1 &', shell=True)
    time.sleep(5)
print(subprocess.run('ollama --version', shell=True, capture_output=True, text=True).stdout)
PULL = subprocess.Popen('ollama pull qwen3.8:27b > /content/ollama_pull.log 2>&1', shell=True)
print('qwen3.8:27b 背景下載中')

In [ ]:
# 1. 下載原始權重（約 54GB）
BASE = '/content/hf/Qwen3.8-27B'
if not os.path.exists(f'{BASE}/model.safetensors.index.json'):
    from huggingface_hub import snapshot_download
    snapshot_download('Qwen/Qwen3.8-27B', local_dir=BASE, max_workers=16)
print('完成', sh(f'du -sh {BASE}', check=False))

In [ ]:
# 2. 重新訓練摘要 LoRA（和上次完全相同的資料與設定；只算回答段的 loss，顯示卡記憶體峰值約 52GB）
ADAPTER = f'{W}/run/final'
if not os.path.exists(f'{ADAPTER}/adapter_model.safetensors'):
    sh(f'{sys.executable} {W}/train_lora.py --base {BASE} --data {W}/data --out {W}/run --max-len 16384 --load-4bit '
       f'--epochs 3 --grad-accum 4 --save-steps 5 --dev-limit 3 --max-train 0')
print(open(f'{W}/run/train_args.json').read())

In [ ]:
# 3. 把 LoRA 加回原始權重（檔名、張量名稱、格式和官方完全一樣）
MERGED = '/content/merged'
if not os.path.exists(f'{MERGED}/model.safetensors.index.json'):
    sh(f'{sys.executable} {W}/merge_lora.py --base {BASE} --adapter {ADAPTER} --out {MERGED}')
sh(f'du -sh {MERGED}; df -h /content | tail -1', check=False)

In [ ]:
# 4. 匯入 Ollama（模型名稱 meetsum-summary）：沿用原版的 Modelfile（對話樣板、參數），只換掉權重
NAME = 'meetsum-summary'
PULL.wait()
print('下載結果：', (open('/content/ollama_pull.log', errors='replace').read()[-300:].replace('\r', '\n').strip().splitlines() or [''])[-1], '（exit', PULL.returncode, '）')
show = subprocess.run('ollama show qwen3.8:27b', shell=True, capture_output=True, text=True).stdout
print(show)
import re
q = (re.search(r'quantization\s+(\S+)', show) or [None, 'Q4_K_M'])[1]
QUANT = {'Q4_K_M': 'q4_K_M', 'Q4_K_S': 'q4_K_S', 'Q8_0': 'q8_0'}.get(q.upper(), 'q4_K_M')
base_mf = subprocess.run('ollama show qwen3.8:27b --modelfile', shell=True, capture_output=True, text=True).stdout
def quick_test(model):
    import requests
    r = requests.post('http://127.0.0.1:11434/api/chat', json={'model': model, 'stream': False, 'format': 'json', 'think': False,
        'messages': [{'role': 'user', 'content': '用 JSON 回答：{"ok": true, "word": "測試"}'}], 'options': {'temperature': 0}}, timeout=900)
    print('快速測試', model, r.status_code, r.text[:300])
    return r.status_code == 200 and '"ok"' in r.text
def make(from_path):
    mf = re.sub(r'^FROM .*$', f'FROM {from_path}', base_mf, count=1, flags=re.M)
    mf = re.sub(r'^ADAPTER .*$\n?', '', mf, flags=re.M)
    open(f'{W}/Modelfile', 'w').write(mf)
    return sh(f'cd {W} && ollama create {NAME} --quantize {QUANT} -f Modelfile', check=False) == 0
ROUTE = None
# 做法 A：讓 Ollama 自己讀合併好的 safetensors 並量化成和原版相同的格式
if make(MERGED) and quick_test(NAME):
    ROUTE = 'A'
else:
    print('--- 做法 A 沒成功，Ollama 紀錄最後 60 行 ---')
    print(''.join(open('/content/ollama.log', errors='replace').readlines()[-60:]))
    # 做法 B：llama.cpp 轉 GGUF（bf16）→ 量化 → 匯入
    LC = '/content/llama.cpp'
    if not os.path.exists(LC):
        sh(f'git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {LC}')
    sh(f'{sys.executable} -m pip install -q -r {LC}/requirements/requirements-convert_hf_to_gguf.txt', check=False)
    sh(f'cd {LC} && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 12 > /dev/null')
    sh(f'{sys.executable} {LC}/convert_hf_to_gguf.py {MERGED} --outfile /content/merged-bf16.gguf --outtype bf16')
    sh(f'{LC}/build/bin/llama-quantize /content/merged-bf16.gguf /content/meetsum-summary-{q}.gguf {q.upper()}')
    os.remove('/content/merged-bf16.gguf')
    if make(f'/content/meetsum-summary-{q}.gguf') and quick_test(NAME):
        ROUTE = 'B'
if ROUTE is None:
    print(''.join(open('/content/ollama.log', errors='replace').readlines()[-80:]))
    raise RuntimeError('兩種做法都沒辦法讓 Ollama 正常執行，請看上面的紀錄')
print('✅ 匯入成功，做法', ROUTE, '，量化格式', QUANT)
sh('ollama list', check=False)

In [ ]:
# 5. 用覓聲正式的呼叫方式實測（8 場：中文 5、英文 3）
#    原版 = qwen3.8:27b + 思考模式（目前 .env 的設定）；微調版 = meetsum-summary + 不思考（訓練時的設定）
EV = f'{W}/eval'
sh(f'{sys.executable} {W}/ollama_eval.py --data {W}/test8.jsonl --model {NAME} --tag ft_ollama --think false --out {EV}')
sh('ollama stop meetsum-summary', check=False)
sh(f'{sys.executable} {W}/ollama_eval.py --data {W}/test8.jsonl --model qwen3.8:27b --tag base_ollama --think true --out {EV}')
B = json.load(open(f'{EV}/base_ollama.report.json')); F = json.load(open(f'{EV}/ft_ollama.report.json'))
HF = json.load(open(f'{W}/hf_sum_report.json'))
names = {'json_ok': 'JSON 合法', 'quote_ok': '引用正確', 'count_diff': '項目數差距（越小越好）', 'topic_f1': '議題吻合',
         'action_f1': '待辦吻合', 'text_sim': '文字相似度', 'headline_sim': '一句話總結相似度'}
for grp in ['all', 'zh', 'en']:
    print(f'\n== {grp}（{B[grp]["n"]} 場）       原版Ollama  微調Ollama  （參考：微調HF）')
    for k, label in names.items():
        print(f'{label:<14} {B[grp][k]!s:>10} {F[grp][k]!s:>10}   {HF["ft"][grp][k]!s:>8}')
print('\n平均每場秒數：原版', B['avg_secs'], '／微調版', F['avg_secs'])

In [ ]:
# 6. 結果打包下載（Ollama 模型檔太大，留在這台主機；要長期保存再另外處理）
import zipfile
mf = subprocess.run(f'ollama show {NAME} --modelfile', shell=True, capture_output=True, text=True).stdout
open(f'{EV}/Modelfile.txt', 'w').write(mf)
Z = '/content/meetsum_summary_ollama_eval.zip'
with zipfile.ZipFile(Z, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{EV}/*'):
        z.write(f, os.path.basename(f))
    for f in [f'{W}/run/train_log.json', f'{W}/run/train_args.json', f'{ADAPTER}/adapter_config.json']:
        if os.path.exists(f):
            z.write(f, 'run/' + os.path.basename(f))
print('===REPORT_BASE==='); print(json.dumps(B, ensure_ascii=False))
print('===REPORT_FT==='); print(json.dumps(F, ensure_ascii=False))
print('===END===')
from google.colab import files
files.download(Z)

In [ ]:
# 7. 關機（確認上面都完成之後才執行）
from google.colab import runtime
runtime.unassign()